In [ ]:
cd C:\R_projects\mothz\detectron2\custom_training

In [ ]:
import torch, detectron2, cv2
!nvcc --version
print("\nTorch CUDA version:", torch.version.cuda)
print("\nTorch version:", torch.__version__)
print("\ndetectron2:", detectron2.__version__)
print("\ncv2 version:",cv2.__version__)


In [ ]:
print("Python version: ")
!python --version

In [ ]:
# Some basic setup:
# Setup detectron2 logger
from detectron2.utils.logger import setup_logger
setup_logger()
import numpy as np
import os, json, cv2, random
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor,DefaultTrainer
from detectron2.config import get_cfg
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.utils.visualizer import ColorMode,GenericMask,Visualizer
from detectron2.structures.keypoints import heatmaps_to_keypoints
from matplotlib import pyplot as plt
import glob
from detectron2.data import build_detection_test_loader
from detectron2.evaluation import (
    CityscapesInstanceEvaluator,
    CityscapesSemSegEvaluator,
    COCOEvaluator,
    COCOPanopticEvaluator,
    LVISEvaluator,
    PascalVOCDetectionEvaluator,
    SemSegEvaluator,
    DatasetEvaluator,
    inference_on_dataset,
    print_csv_format,
    verify_results,
)

In [ ]:
from detectron2.custom.misc import *
from detectron2.custom.parse_ruler_tags import *

In [ ]:
# Register datasets and populate metadata
register_coco_instances("mothz_mask_train", {}, ".\mask\coco_mask_train.json", "C:\R_projects\mothz\coco-annotator\datasets\mothz_sample1")
register_coco_instances("mothz_mask_test", {}, ".\mask\coco_mask_test.json", "C:\R_projects\mothz\coco-annotator\datasets\mothz_sample1")
register_coco_instances("mothz_mask_val", {}, ".\mask\coco_mask_val.json", "C:\R_projects\mothz\coco-annotator\datasets\mothz_sample1")

In [ ]:
things = ["body","hindwing","color_checker","ruler", "tag"] #["body", "forewing", "hindwing", "whole_moth", "color_checker", "ruler", "tag"]
things_color = [(255, 0, 0), (0, 255, 0),(0, 0, 255),(100, 100, 100),(255, 255, 0)]

MetadataCatalog.get("mothz_mask_train").set(thing_classes=things)
MetadataCatalog.get("mothz_mask_train").set(thing_colors=things_color)

MetadataCatalog.get("mothz_mask_test").set(thing_classes=things)
MetadataCatalog.get("mothz_mask_test").set(thing_colors=things_color)

MetadataCatalog.get("mothz_mask_val").set(thing_classes=things)
MetadataCatalog.get("mothz_mask_val").set(thing_colors=things_color)

In [ ]:
# Set model configs
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("mothz_mask_train",)
cfg.DATASETS.TEST = ("mothz_mask_val", ) # Called test in detectron2, but is actually the validation data
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")  # Let training initialize from model zoo
cfg.DATALOADER.NUM_WORKERS = 2
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS = False 
cfg.TEST.EVAL_PERIOD = 0 # Turn off COCO evluator. 
cfg.SOLVER.IMS_PER_BATCH = 2  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.0001  
cfg.SOLVER.MAX_ITER = 2500    
cfg.SOLVER.LR_SCHEDULER_NAME = "WarmupCosineLR"
cfg.SOLVER.WARMUP_ITERS = 100
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128   # The "RoIHead batch size"
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(things) # Change for the number of classes
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.03 # Set to lower?
cfg.TEST.DETECTIONS_PER_IMAGE = 20 # Set higher to allow filtering by class later
cfg.SOLVER.CHECKPOINT_PERIOD = 200
cfg.MODEL.ROI_KEYPOINT_HEAD.NUM_KEYPOINTS = 0
cfg.TEST.KEYPOINT_OKS_SIGMAS = []
cfg.INPUT.RANDOM_FLIP = "horizontal"
cfg.MODEL.MASK_ON = True
cfg.MODEL.KEYPOINT_ON = False # Keypoint detection for multiple thing classes not supported by detectron2
cfg.MODEL.ROI_KEYPOINT_HEAD.LOSS_WEIGHT = 5
cfg.MODEL.ROI_KEYPOINT_HEAD.NORMALIZE_LOSS_BY_VISIBLE_KEYPOINTS = False
cfg.OUTPUT_DIR = "./mask/model_v1"
#cfg.MODEL.BACKBONE.FREEZE_AT = 0
cfg.INPUT.MAX_SIZE_TEST = 2000
cfg.INPUT.MAX_SIZE_TRAIN = 2000
cfg.INPUT.MIN_SIZE_TEST = 2000
cfg.INPUT.MIN_SIZE_TRAIN = 2000

# cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN = 5000
#cfg.MODEL.RPN.PRE_NMS_TOPK_TEST = 3000
#cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 2500
#cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 2000
cfg.MODEL.RPN.NMS_THRESH = 0.9


os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

In [ ]:
print(cfg)

In [ ]:
### Train model
trainer = CustomTrainer(cfg)
trainer.resume_or_load(resume=True)
trainer.train()

In [ ]:
# Look at model performance over iterations
%load_ext tensorboard
#%reload_ext tensorboard
%tensorboard --logdir "C:/R_projects/mothz/detectron2/custom_training/mask/model_v1"

### Go to directroy to delete temp file if TensorBoard fails to launch
# !del /S C:\Users\vsbpa\AppData\Local\Temp\.tensorboard-info

In [ ]:
# Load fitted model as predictor
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_0001999.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3  # set to 0.3 for now, but actually filtered to 0.7 in R
predictor = DefaultPredictor(cfg)

In [ ]:
%matplotlib inline 

f = glob.glob("C:/R_projects/mothz/coco-annotator/datasets/mothz_sample1/1-3-24__subdir__2024_01_03__subdir__IMG_0009_1.JPG")
fi = random.choice(f)
#fi = f[0]
im = cv2.imread(fi)
outputs = predictor(im)  # format is documented at https://detectron2.readthedocs.io/tutorials/models.html#model-output-format
v = Visualizer(im[:, :, 0:3][:,:,::-1],
                   metadata=MetadataCatalog.get("mothz_mask_train"), 
                   scale=0.3, 
               instance_mode=ColorMode.IMAGE_BW
)
outputs = filter_instance_by_classes(outputs, "mothz_mask_test",[1, 2, 1, 1, 4]) # max detection: ['body', 'hindwing', 'color_checker', 'ruler', 'tag']
out = v.draw_instance_predictions(outputs)


plt.figure(figsize = (15,15))
plt.imshow(out.get_image()[:,:,0:3], interpolation='nearest')
plt.title(fi)
plt.show()
print(read_img_tags(outputs, im,  "mothz_mask_train"))
print(ruler_tick(outputs, im,  "mothz_mask_train"))

In [ ]:
# For writing predictions
from detectron2.custom.writing import *

In [ ]:
# Perform inference on the training dataset
write_img_inference(
    predictor, 
    write_path = "C:/R_projects/mothz/",
    name = "full_mothz_sample1",
    read_path = "C:/R_projects/mothz/coco-annotator/datasets/mothz_sample1/", 
    model_ver = "modelv1.0",
    meta = "mothz_mask_train",
    max_detection = [1, 2, 1, 1, 4], # ['body', 'hindwing', 'color_checker', 'ruler', 'tag']
    mode = "mask"
    )

In [ ]:
# Perform inference on all the data
for d in glob.glob("D:/moth_photos/database/*"):
    print("Starting with directory: " + d)
    write_img_inference(
        predictor, 
        write_path = "C:/R_projects/mothz/",
        name = os.path.basename(d),
        read_path = d, 
        model_ver = "modelv1.0",
        meta = "mothz_mask_train",
        max_detection = [1, 2, 1, 1, 4], # ['body', 'hindwing', 'color_checker', 'ruler', 'tag']
        mode = "mask"
    )



In [ ]:
# Perform inference on allie's dataset
write_img_inference(
    predictor, 
    write_path = "C:/R_projects/mothz/",
    name = "allie_historic",
    read_path = "D:/moth_photos/database/batch_08", 
    model_ver = "modelv1.0",
    meta = "mothz_mask_train",
    max_detection = [1, 2, 1, 1, 4], # ['body', 'hindwing', 'color_checker', 'ruler', 'tag']
    mode = "mask"
    )